# 10 — Attention : quelles coupes sont ignorées

Les 24 coupes NPZ sont un **linspace dans la bande 20–80 %**. Chaque tête fait un softmax sur 72 positions (3 plans × 24).

**v11** : 72 vecteurs GAP partagés ; les 12 têtes ne diffèrent que par le score.

**v14** : score de coupe encore sur les 72 GAP ; pooling **72 × 12** tokens CAM 7×7.

**v17** (ce notebook) : même score de coupe sur le GAP, mais un **BiLSTM** (hidden 256, résidu init 0) mélange d’abord les 24 coupes **dans l’ordre**, par plan. Le pooling utilise des tokens CAM **14×14** (layer4 dilated, ~16 px/case). `interpret()` fournit `slice_attn` (GAP + LSTM) et `cam_tokens` (CAM + LSTM). Une cellule plus bas compare au contrefactuel « scorer la coupe sur le token CAM » (piste v15).

Défaut : **full-train v17** gold (`v17_crop130v1`, best epoch 12, val 0,873). `SPLIT = "val"` relance le forward sur ~653 études. Pour le subset 25 % : décommenter l’autre `CONFIG`.

```text
python -X utf8 -m src.infer.inspect_attention --config configs/archive/kneenet_v17_crop130.yaml --split gold --workers 0
python -X utf8 -m src.infer.inspect_attention --config configs/archive/kneenet_v17_crop130.yaml --split val --workers 0
```


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.infer.inspect_attention import load_config, run

CONFIG = ROOT / "configs" / "kneenet_v17_crop130.yaml"
# CONFIG = ROOT / "configs" / "kneenet_v17_crop130_subset.yaml"  # 25 % train
FORCE = False
SPLIT = "gold"  # "val" (~653) | "gold" (58)
MAX_STUDIES = None

cfg = load_config(str(CONFIG))
OUT = ROOT / "checkpoints" / cfg["run_name"] / "attention_inspect"
if SPLIT == "gold":
    OUT = ROOT / "checkpoints" / cfg["run_name"] / "attention_inspect_gold"
CKPT = ROOT / "checkpoints" / cfg["run_name"] / "best.pt"
if not CKPT.exists():
    CKPT = ROOT / "checkpoints" / cfg["run_name"] / "last.pt"
print("config", CONFIG.name)
print("ckpt  ", CKPT, "exists", CKPT.exists())
print("out   ", OUT)


In [ ]:
import torch

summary_path = OUT / "summary.json"
if summary_path.exists() and not FORCE:
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("chargé", summary_path)
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    summary = run(
        cfg, CKPT, SPLIT, OUT, device,
        batch_size=int(cfg.get("batch_size") or 4),
        num_workers=0,
        max_studies=MAX_STUDIES,
        n_examples=4,
    )

print(
    f"{summary['model']}  epoch {summary['epoch']}  "
    f"val AUC {summary['val_macro_auc']}  "
    f"{summary['split']} n={summary['n_studies']}"
)


In [ ]:
tbl = pd.read_csv(OUT / "slice_attention.csv")
sug = summary["suggested"]
dead = summary["dead_idx"]
tbl["flag"] = ""
tbl.loc[tbl.slice.isin(sug["drop_idx"]), "flag"] = "bord 90%"
tbl.loc[tbl.slice.isin(dead), "flag"] = "morte"
display(
    tbl.style.format({
        "depth_pct": "{:.0f}%",
        "mass": "{:.2%}",
        "pct_top5_any": "{:.1%}",
        "mass_sag": "{:.4f}",
        "mass_cor": "{:.4f}",
        "mass_axi": "{:.4f}",
    })
)
print(
    f"90 % de la masse : coupes {sug['keep_lo']}–{sug['keep_hi']} "
    f"({sug['n_keep']}/24) → bande "
    f"{100*sug['band_lo']:.0f}–{100*sug['band_hi']:.0f} % "
    f"(masse {100*sug['kept_mass']:.1f} %)"
)
print("coupes mortes :", dead if dead else "(aucune au sens strict)")
print(
    f"sans reconvertir : x[:, :, {sug['keep_lo']}:{sug['keep_hi']+1}]  "
    f"(D={sug['n_keep']})"
)
print(
    f"reconvertir : BAND=({sug['band_lo']:.2f}, {sug['band_hi']:.2f}) "
    f"et garder 24 coupes"
)

In [ ]:
for name in ("attn_profile.png", "attn_heatmap_planes.png", "attn_heatmap_targets.png"):
    p = OUT / name
    print(p.name)
    display(Image(filename=str(p)))

In [ ]:
ex_dir = OUT / "examples"
files = sorted(ex_dir.glob("*.png")) if ex_dir.exists() else []
print(len(files), "mosaics")
for p in files[:6]:
    print(p.name)
    display(Image(filename=str(p)))

## v17 — 72 GAP (+ LSTM) vs 72×12 tokens CAM

Le **forward v17** score encore les 72 GAP (après le résidu BiLSTM), puis pondère les tokens CAM 14×14. Les 12× plus de vecteurs changent donc le **contenu poolé**, pas le **choix de coupe** — sauf si le LSTM a déjà mélangé l’information des voisines dans le GAP.

La cellule suivante affiche le **contrefactuel** calculé pendant le run : passer `attn_score` (entraîné sur GAP+LSTM) sur le token CAM de chaque tête, puis softmax coupe. Ce n’est **pas** le modèle entraîné — aperçu de ce que ferait un v15 qui scorerait les tokens CAM.


In [ ]:
CMP_CSV = OUT / "gap_vs_cam_slice.csv"
if not CMP_CSV.exists():
    print("pas encore de gap_vs_cam_slice.csv — relancer la cellule 2 avec FORCE=True")
else:
    cmp_tbl = pd.read_csv(CMP_CSV)
    print(CMP_CSV)
    print(
        f"corr mediane {cmp_tbl['corr'].median():.3f}  "
        f"pics differents {(cmp_tbl.peak_gap != cmp_tbl.peak_cam).sum()}/{len(cmp_tbl)}"
    )
    display(cmp_tbl.round(3))
    for name in ("gap_vs_cam_slice.png", "gap_vs_cam_heatmaps.png"):
        p = OUT / name
        if p.exists():
            display(Image(filename=str(p)))


## Lecture (v17 full — gold 58, epoch 12)

`v17_crop130v1` / `best.pt` : val 0,873, gold 0,906. Softmax coupe **moins plat** qu’en v14 subset (masse 2,2–6,3 % vs 3,7–4,6 % ; uniforme = 4,17 %). **Aucune coupe morte.** Le BiLSTM concentre un peu (pics 5, 11, 16) sans tuer de positions.

La coupe **6** (36 %) est la plus pauvre (masse 2,2 %, top-5 13 %) mais elle est **interne** : `trim_band` ne la voit pas. L’auto-trim « drop 21–23 » n’enlève que des bords encore à ~3 % : **ne pas retirer d’indices NPZ ni resserrer BAND**.

Plans : sag 40,3 % / cor 35,3 % / ax 24,3 % — axial proche du v11 full (~22 %), plus bas que le v14 subset (~30 %). Pics encore anatomiques : ACL coupe 20 (~72 %), ménisque médial ~13, ménisque latéral / MCL ~16, OA médial ~5, épanchement ~3, Baker ~4.

**72×12 tokens :** le forward score toujours GAP+LSTM. Le contrefactuel « scorer le token CAM » (médiane corr **0,74**, 9/12 pics différents) diverge plus qu’en v14 (corr 0,91–0,94). Accord encore correct pour MCL / OA médial / Baker ; plus faible pour ACL (0,64), épanchement (0,59), contusion (0,60), fracture (0,57). Scorer les tokens CAM n’est utile que si on **entraîne** `attn_score` dessus (v15), pas en réutilisant les poids GAP.
